In [1]:
import pathlib

import torch

from artist.data_parser import paint_scenario_parser
from artist.scenario.configuration_classes import (
    LightSourceConfig,
    LightSourceListConfig,
)
from artist.scenario.h5_scenario_generator import H5ScenarioGenerator
from artist.util import config_dictionary, set_logger_config
from artist.util.environment_setup import get_device

# Set up logger
set_logger_config()

torch.manual_seed(7)
torch.cuda.manual_seed(7)

# Set the device.
device = get_device()


[2025-12-26 02:06:05,089][artist.util.environment_setup][INFO] - No device type provided. The device will default to GPU based on availability and OS, otherwise to CPU.
[2025-12-26 02:06:05,090][artist.util.environment_setup][WARNING] - Setting device to CPU. ARTIST only supports CPU for MacOS.


In [2]:
def get_the_latest_deflectometry_path(deflectometry_dir):
    from datetime import datetime
    
    filled_files = []
    
    for path in pathlib.Path(deflectometry_dir).glob("*.h5"):
        # print(path.name)
        if "-filled-" in path.name:
            
            # Extract date from filename (format: YYYY-MM-DDZHH-MM-SSZ)
            try:
                # Split by "-filled-" and get the date part
                date_part = path.name.split("-filled-")[1].split("Z-")[0]
                # print(f"Extracted date part: {date_part}")

                filled_files.append((date_part, path))
            except (ValueError, IndexError):
                print(f"Could not parse date from filename: {path.name}")
                continue
    
    if not filled_files:
        return None
    
    # Sort by date and return the path with the newest date
    filled_files.sort(key=lambda x: x[0], reverse=True)
    return filled_files[0][1]


def get_heliostat_properties_path(heliostat_dir):
    for path in pathlib.Path(heliostat_dir).glob("*.json"):
        if "properties" in path.name:
            return path
    return None

In [3]:
print(get_the_latest_deflectometry_path("paint_data/AA31/Deflectometry"))

paint_data/AA31/Deflectometry/AA31-filled-2023-09-18Z11-50-48Z-deflectometry.h5


In [4]:
def generate_single_scenario(scenario_path: pathlib.Path, tower_file: pathlib.Path, heliostat_files_list: list):

    # Specify the path where you want to save the scenario file.
    # scenario_path = pathlib.Path("please/insert/the/path/to/the/scenario/here/name")

    # Specify the path to your tower-measurements.json file.
    # tower_file = pathlib.Path(
    #     "please/insert/the/path/to/the/paint/data/here/tower-measurements.json"
    # )

    # Specify the following data for each heliostat that you want to include in the scenario:
    # A tuple of: (heliostat-name, heliostat-properties.json, deflectometry.h5)
    # heliostat_files_list = [
    #     (
    #         "name",
    #         pathlib.Path(
    #             "please/insert/the/path/to/the/paint/data/here/heliostat-properties.json"
    #         ),
    #         pathlib.Path("please/insert/the/path/to/the/paint/data/here/deflectometry.h5"),
    #     ),
    #     (
    #         "name2",
    #         pathlib.Path(
    #             "please/insert/the/path/to/the/paint/data/here/heliostat-properties.json"
    #         ),
    #         pathlib.Path("please/insert/the/path/to/the/paint/data/here/deflectometry.h5"),
    #     ),
    #     # ... Include as many as you want, but at least one!
    # ]

    # This checks to make sure the path you defined is valid and a scenario HDF5 can be saved there.
    if not pathlib.Path(scenario_path).parent.is_dir():
        # Create the parent directories if they do not exist.
        print(f"The path to the scenario does not exist. Creating directories for scenario path: {scenario_path}")
        pathlib.Path(scenario_path).parent.mkdir(parents=True, exist_ok=True)
        if(not pathlib.Path(scenario_path).parent.is_dir()):
            raise FileNotFoundError(
                f"The specified scenario path {scenario_path} is not valid and could not be created."
            )
    else:
        print(f"The scenario will be saved to: {scenario_path}")
        
    # Include the power plant configuration.
    power_plant_config, target_area_list_config = (
        paint_scenario_parser.extract_paint_tower_measurements(
            tower_measurements_path=tower_file, device=device
        )
    )

    # Include the light source configuration.
    light_source1_config = LightSourceConfig(
        light_source_key="sun_1",
        light_source_type=config_dictionary.sun_key,
        number_of_rays=10,
        distribution_type=config_dictionary.light_source_distribution_is_normal,
        mean=0.0,
        covariance=4.3681e-06,
    )

    # Create a list of light source configs - in this case only one.
    light_source_list = [light_source1_config]

    # Include the configuration for the list of light sources.
    light_source_list_config = LightSourceListConfig(light_source_list=light_source_list)

    target_area = [
        target_area
        for target_area in target_area_list_config.target_area_list
        if target_area.target_area_key == config_dictionary.target_area_receiver
    ]

    number_of_nurbs_control_points = torch.tensor([20, 20], device=device)
    nurbs_fit_method = config_dictionary.fit_nurbs_from_normals
    nurbs_deflectometry_step_size = 100
    nurbs_fit_tolerance = 1e-10
    nurbs_fit_max_epoch = 400

    # Please leave the optimizable parameters empty, they will automatically be added for the surface fit.
    nurbs_fit_optimizer = torch.optim.Adam([torch.empty(1, requires_grad=True)], lr=1e-3)
    nurbs_fit_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        nurbs_fit_optimizer,
        mode="min",
        factor=0.2,
        patience=50,
        threshold=1e-7,
        threshold_mode="abs",
    )

    heliostat_list_config, prototype_config = (
        paint_scenario_parser.extract_paint_heliostats_fitted_surface(
            paths=heliostat_files_list,
            power_plant_position=power_plant_config.power_plant_position,
            number_of_nurbs_control_points=number_of_nurbs_control_points,
            deflectometry_step_size=nurbs_deflectometry_step_size,
            nurbs_fit_method=nurbs_fit_method,
            nurbs_fit_tolerance=nurbs_fit_tolerance,
            nurbs_fit_max_epoch=nurbs_fit_max_epoch,
            nurbs_fit_optimizer=nurbs_fit_optimizer,
            nurbs_fit_scheduler=nurbs_fit_scheduler,
            device=device,
        )
    )

    """Generate the scenario given the defined parameters."""
    scenario_generator = H5ScenarioGenerator(
        file_path=scenario_path,
        power_plant_config=power_plant_config,
        target_area_list_config=target_area_list_config,
        light_source_list_config=light_source_list_config,
        prototype_config=prototype_config,
        heliostat_list_config=heliostat_list_config,
    )
    scenario_generator.generate_scenario()


In [5]:

# for all the heliostats in the paint_data folder, create a scenario for each individual calibration reading
data_path = pathlib.Path("./paint_data")

tower_measurement_file = "./paint_data/tower-measurements.json"

scenario_root_folder = pathlib.Path("./scenarios/one_heliostat_scenarios")
#iterate through each heliostat folder
for heliostat_folder in data_path.iterdir():
    if heliostat_folder.is_dir():
        heliostat_name = heliostat_folder.name
        print(f"Processing heliostat: {heliostat_name}")
        
        defletctometry_folder_path = heliostat_folder / "Deflectometry"
        latest_deflectometry_path = get_the_latest_deflectometry_path(defletctometry_folder_path)

        heliostat_properties_folder = heliostat_folder / "Properties"
        heliostat_properties_path = get_heliostat_properties_path(heliostat_properties_folder)
        
        if( heliostat_properties_path is None ):
            print(f"No heliostat properties file found for {heliostat_name}, skipping.")
            continue

        if latest_deflectometry_path is None:
            print(f"No filled deflectometry files found for {heliostat_name}, skipping.")
            continue
        calibration_folder = heliostat_folder / "Calibration"
        if not calibration_folder.exists():
            print(f"No Calibration folder found for {heliostat_name}, skipping.")
            continue
        

        scenario_file_path = scenario_root_folder / f"{heliostat_name}_one_heliostat_scenario.h5"

        heliostat_files_list = [
                (
                    "name",
                    pathlib.Path(
                        heliostat_properties_path
                    ),
                    pathlib.Path(latest_deflectometry_path),
                ),
            ]

        generate_single_scenario(scenario_path=scenario_file_path, tower_file= tower_measurement_file, heliostat_files_list=heliostat_files_list)
        print(f"DONE: Scenario for heliostat {heliostat_name} generated and saved to {scenario_file_path}.\n")

Processing heliostat: AA31
The scenario will be saved to: scenarios/one_heliostat_scenarios/AA31_one_heliostat_scenario.h5
[2025-12-26 02:06:05,112][artist.data_parser.paint_scenario_parser][INFO] - Beginning extraction of tower data from PAINT file.
[2025-12-26 02:06:05,114][artist.data_parser.paint_scenario_parser][INFO] - Loading tower data complete.
[2025-12-26 02:06:05,603][artist.data_parser.paint_scenario_parser][INFO] - Beginning extraction of heliostat properties data from PAINT file.
[2025-12-26 02:06:05,604][artist.data_parser.paint_scenario_parser][INFO] - Loading heliostat properties data complete.
[2025-12-26 02:06:05,604][artist.data_parser.paint_scenario_parser][INFO] - Beginning extraction of deflectometry data from PAINT file.
[2025-12-26 02:06:06,946][artist.data_parser.paint_scenario_parser][INFO] - Loading deflectometry data complete.
[2025-12-26 02:06:06,947][artist.scenario.surface_generator][INFO] - Beginning generation of the fitted surface configuration.
[2025